# 

Aşama 2: PyTorch ile Zaman Serisi Satış Tahmini (LSTM vs GRU)

Bu projede Rossmann Mağaza Satışları veri setini kullanarak gelecekteki mağaza satışlarını (regresyon problemi) tahmin ediyoruz.
Train/Test ayrımı %80 Eğitim, %20 Test olacak şekilde güncellenmiştir.

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import time
import random

# PyTorch için device ayarı
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Kullanılan cihaz: {device}')

# Seed ayarlama (her çalıştırmada aynı sonucu almak için)
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


Kullanılan cihaz: cuda


## 1. Veri Yükleme ve Ön İşleme

In [2]:
import pandas as pd
import numpy as np

# Veriyi yükleme
df = pd.read_csv('train.csv', low_memory=False)
store_df = pd.read_csv('store.csv', low_memory=False)

# Sadece açık olan mağazaları filtreleme (Kapalıysa satış zaten 0'dır)
df = df[df['Open'] == 1]

# Store verisini ana veri setine ekliyoruz (MERGE)
df = df.merge(store_df, on='Store', how='left')

# Tarih sütununu datetime formatına çevirme
df['Date'] = pd.to_datetime(df['Date'])

# Tarihe ve mağazaya göre sıralama
df = df.sort_values(by=['Store', 'Date'])
df


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
843264,1,3,2013-01-02,5530,668,1,0,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
842155,1,4,2013-01-03,4327,578,1,0,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
841047,1,5,2013-01-04,4486,619,1,0,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
839940,1,6,2013-01-05,4997,635,1,0,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
838815,1,1,2013-01-07,7176,785,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5564,1115,1,2015-07-27,10712,608,1,1,0,1,d,c,5350.0,NaN,NaN,1,22.0,2012.0,"Mar,Jun,Sept,Dec"
4451,1115,2,2015-07-28,8093,500,1,1,0,1,d,c,5350.0,NaN,NaN,1,22.0,2012.0,"Mar,Jun,Sept,Dec"
3338,1115,3,2015-07-29,7661,473,1,1,0,1,d,c,5350.0,NaN,NaN,1,22.0,2012.0,"Mar,Jun,Sept,Dec"
2225,1115,4,2015-07-30,8405,502,1,1,0,1,d,c,5350.0,NaN,NaN,1,22.0,2012.0,"Mar,Jun,Sept,Dec"


## 2. Kayan Pencere (Sliding Window) ve Veri Ölçekleme
Veri seti %80 Eğitim (Train) ve %20 Test olarak bölünecek şekilde ayarlanmıştır.

In [3]:
from sklearn.preprocessing import MinMaxScaler

# 1. Mevcut öznitelikler
df['DayOfWeek_Scaled'] = (df['DayOfWeek'] - 1) / 6.0
df['Promo'] = df['Promo'].astype(float)
df['SchoolHoliday'] = df['SchoolHoliday'].astype(float)

# Yeni Eklenen Özellikler: Ay, Gün ve Tatil
df['Month_Scaled'] = (df['Date'].dt.month - 1) / 11.0
df['Day_Scaled'] = (df['Date'].dt.day - 1) / 30.0
df['StateHoliday'] = df['StateHoliday'].astype(str)
df['Sales_Rolling_7'] = df.groupby('Store')['Sales'].transform(lambda x: x.rolling(7, min_periods=1).mean())
roll_scaler = MinMaxScaler()
df['Sales_Rolling_7_Scaled'] = roll_scaler.fit_transform(df[['Sales_Rolling_7']])


df = pd.get_dummies(df, columns=['StateHoliday'], dtype=float)
state_holiday_cols = [c for c in df.columns if 'StateHoliday_' in c]

# 2. Yeni eklenen Store özellikleri
# CompetitionDistance (Rakip uzaklığı): Boş olanları medyan ile dolduralım
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(df['CompetitionDistance'].median())

# StoreType ve Assortment (Metinsel/Kategorik veriler) One-Hot Encoding ile 0 ve 1'lere çevrilir
df = pd.get_dummies(df, columns=['StoreType', 'Assortment'], dtype=float)

# Hangi yeni kategorik sütunların oluştuğunu bulalım (örn: StoreType_a, Assortment_c vb.)
store_type_cols = [c for c in df.columns if 'StoreType_' in c]
assortment_cols = [c for c in df.columns if 'Assortment_' in c]

# Promo2 zaten 0 ve 1'den oluşuyor
df['Promo2'] = df['Promo2'].astype(float)

# Satışları ve Rakip Uzaklığını Ölçeklendirme
scaler = MinMaxScaler()
df['Sales_Scaled'] = scaler.fit_transform(df[['Sales']])

dist_scaler = MinMaxScaler()
df['CompetitionDistance_Scaled'] = dist_scaler.fit_transform(df[['CompetitionDistance']])

# Tüm Özellikleri Birleştirme
scaled_features = ['Sales_Scaled', 'Promo', 'DayOfWeek_Scaled', 'SchoolHoliday', 'CompetitionDistance_Scaled', 'Promo2', 'Month_Scaled', 'Day_Scaled'] + store_type_cols + assortment_cols + ['Sales_Rolling_7_Scaled'] + state_holiday_cols

def create_sequences(data_grouped, seq_length):
    xs, ys = [], []
    for store_id, group in data_grouped:
        store_data = group[scaled_features].values
        if len(store_data) <= seq_length:
            continue
        for i in range(len(store_data) - seq_length):
            xs.append(store_data[i:(i + seq_length)])
            ys.append(store_data[i + seq_length, 0]) # Sales_Scaled her zaman 0. indekste
    return np.array(xs), np.array(ys)

# Veriyi 2013-2014 Eğitim, 2015 Test olacak şekilde ayırıyoruz
split_date = pd.to_datetime('2015-01-02')

train_df = df[df['Date'] < split_date]
test_df = df[df['Date'] >= split_date]

# Zaman penceresi
seq_length = 45
print('Tensörler oluşturuluyor... (Birkaç dakika sürebilir)')

X_train, y_train = create_sequences(train_df.groupby('Store'), seq_length)
X_test, y_test = create_sequences(test_df.groupby('Store'), seq_length)

print('Eğitim seti (%80):', X_train.shape, y_train.shape)
print('Test seti (%20):', X_test.shape, y_test.shape)
print('Kullanılan Özellik Sayısı:', len(scaled_features))
input_size_dynamic = len(scaled_features)


Tensörler oluşturuluyor... (Birkaç dakika sürebilir)
Eğitim seti (%80): (598203, 45, 20) (598203,)
Test seti (%20): (145839, 45, 20) (145839,)
Kullanılan Özellik Sayısı: 20


In [4]:
batch_size = 512 
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(-1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(-1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)


In [5]:
import torch.nn as nn

class SalesLSTM(nn.Module):
    def __init__(self, input_size=input_size_dynamic, hidden_size=128, num_layers=2):
        super(SalesLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2 if num_layers > 1 else 0)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

class SalesGRU(nn.Module):
    def __init__(self, input_size=input_size_dynamic, hidden_size=128, num_layers=2):
        super(SalesGRU, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2 if num_layers > 1 else 0)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.gru(x)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out


In [6]:
def train_model(model, train_loader, val_loader=None, epochs=100, lr=0.001, patience=15):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=25, gamma=0.5)
    
    start_time = time.time()
    best_val_loss = float('inf')
    epochs_no_improve = 0
    import copy
    best_model_state = copy.deepcopy(model.state_dict())
    
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        scheduler.step()
        train_loss = epoch_loss/len(train_loader)
        
        if val_loader is not None:
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch_x, batch_y in val_loader:
                    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                    outputs = model(batch_x)
                    v_loss = criterion(outputs, batch_y)
                    val_loss += v_loss.item()
            val_loss = val_loss / len(val_loader)
            
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}')
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                best_model_state = copy.deepcopy(model.state_dict())
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f"\n--- EARLY STOPPING (Erken Durdurma) Tetiklendi! ---")
                    print(f"Son {patience} turdur test verisinde iyilesme yok. Egitim Epoch {epoch+1}'de durduruldu.")
                    print(f"Modelin en basarili haline (Epoch {epoch+1-patience}) geri donuluyor...")
                    model.load_state_dict(best_model_state)
                    # We truncate the losses to the best epoch so the graphs/checks reflect the final model
                    train_losses = train_losses[:-(patience-1)]
                    val_losses = val_losses[:-(patience-1)]
                    break
        else:
            train_losses.append(train_loss)
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {train_loss:.6f}')
            
    end_time = time.time()
    training_time = end_time - start_time
    print(f'Egitim tamamlandi! Sure: {training_time:.2f} sn')
    return model, training_time, train_losses, val_losses

In [7]:
print("\n--- LSTM MODEL EGITIMI (Lutfen Bekleyin) ---")
lstm_model = SalesLSTM(hidden_size=512)
lstm_model, lstm_time, lstm_tl, lstm_vl = train_model(lstm_model, train_loader, test_loader, epochs=100)



--- LSTM MODEL EGITIMI (Lutfen Bekleyin) ---


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:


print("\n--- GRU MODEL EĞİTİMİ (Lütfen Bekleyin) ---")
gru_model = SalesGRU(hidden_size=512)
gru_model, gru_time, gru_tl, gru_vl = train_model(gru_model, train_loader, test_loader, epochs=100)

import xgboost as xgb
import time

print("\n--- XGBoost MODEL EĞİTİMİ (Lütfen Bekleyin) ---")
start_time = time.time()
X_train_xgb = X_train.reshape(X_train.shape[0], -1)
y_train_xgb = y_train
X_test_xgb = X_test.reshape(X_test.shape[0], -1)

# GPU destekli hızlı XGBoost modeli
xgb_device = 'cuda' if torch.cuda.is_available() else 'cpu'
xgb_model = xgb.XGBRegressor(n_estimators=1000, max_depth=9, learning_rate=0.02, subsample=0.9, colsample_bytree=0.8, tree_method='hist', device=xgb_device)
xgb_model.fit(X_train_xgb, y_train_xgb)
xgb_time = time.time() - start_time
print(f'XGBoost Eğitimi tamamlandı! Süre: {xgb_time:.2f} sn')


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import torch

def evaluate_model(model, test_loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch_x, _ in test_loader:
            batch_x = batch_x.to(device)
            outputs = model(batch_x)
            predictions.extend(outputs.cpu().numpy())
    
    predictions = scaler.inverse_transform(predictions)
    return predictions

# Sifira bolme hatasini engelleyen guvenli MAPE fonksiyonu
def safe_mape(y_true, y_pred):
    mask = y_true != 0
    return (np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])).mean() * 100

lstm_preds = evaluate_model(lstm_model, test_loader).flatten()
gru_preds = evaluate_model(gru_model, test_loader).flatten()
y_true = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

xgb_preds_scaled = xgb_model.predict(X_test_xgb)
xgb_preds = scaler.inverse_transform(xgb_preds_scaled.reshape(-1, 1)).flatten()


# LSTM Metrikleri
lstm_mse = mean_squared_error(y_true, lstm_preds)
lstm_rmse = np.sqrt(lstm_mse)
lstm_mae = mean_absolute_error(y_true, lstm_preds)
lstm_r2 = r2_score(y_true, lstm_preds)
lstm_mape = safe_mape(y_true, lstm_preds)

# GRU Metrikleri
gru_mse = mean_squared_error(y_true, gru_preds)
gru_rmse = np.sqrt(gru_mse)
gru_mae = mean_absolute_error(y_true, gru_preds)
gru_r2 = r2_score(y_true, gru_preds)
gru_mape = safe_mape(y_true, gru_preds)

# XGBoost Metrikleri
xgb_mse = mean_squared_error(y_true, xgb_preds)
xgb_rmse = np.sqrt(xgb_mse)
xgb_mae = mean_absolute_error(y_true, xgb_preds)
xgb_r2 = r2_score(y_true, xgb_preds)
xgb_mape = safe_mape(y_true, xgb_preds)


results_df = pd.DataFrame({
    'Model': ['LSTM', 'GRU', 'XGBoost'],
    'MSE': [lstm_mse, gru_mse, xgb_mse],
    'RMSE': [lstm_rmse, gru_rmse, xgb_rmse],
    'MAE': [lstm_mae, gru_mae, xgb_mae],
    'MAPE (%)': [lstm_mape, gru_mape, xgb_mape],
    'R2 Skoru': [lstm_r2, gru_r2, xgb_r2],
    'Eğitim Süresi (sn)': [lstm_time, gru_time, xgb_time]
})

print("\n--- LSTM ve GRU Modellerinin Karşılaştırması ---")
print(results_df.to_string(index=False))




In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))
plt.plot(y_true[:100], label='Gerçek Satışlar', marker='o', linewidth=2)
plt.plot(lstm_preds[:100], label='LSTM Tahminleri', marker='x', alpha=0.8)
plt.plot(gru_preds[:100], label='GRU Tahminleri', marker='^', alpha=0.8)
plt.plot(xgb_preds[:100], label='XGBoost Tahminleri', marker='s', alpha=0.8, color='red', linestyle='--')
plt.title('Zaman Serisi Tahmini (İlk 100 Gün Karşılaştırması)')
plt.xlabel('Zaman Adımı (Gün)')
plt.ylabel('Satış Miktarı')
plt.legend()
plt.grid(True)
plt.savefig('grafik_sonucu.png') # Grafigi bilgisayara da kaydet!
plt.show()


In [ ]:

from IPython.display import display, Markdown

dynamic_report = f"""
### Nihai Tahmin Performansı ve Model Karşılaştırması (2015 Test Seti)

Aşağıdaki tabloda; PyTorch tabanlı Derin Öğrenme modellerimiz (LSTM ve GRU) ile Ağaç tabanlı Makine Öğrenmesi modelimizin (XGBoost) test verisi üzerindeki nihai hata payları ve doğruluk skorları yer almaktadır:

| Model | MSE | RMSE | MAE | MAPE (Hata Payı) | R2 Skoru (Başarı) | Eğitim Süresi |
|-------|-----|------|-----|------------------|-------------------|---------------|
| **LSTM** | {lstm_mse:.0f} | {lstm_rmse:.2f} | {lstm_mae:.2f} | %{lstm_mape:.2f} | **%{lstm_r2*100:.2f}** | {lstm_time:.1f} sn |
| **GRU**  | {gru_mse:.0f} | {gru_rmse:.2f} | {gru_mae:.2f} | %{gru_mape:.2f} | **%{gru_r2*100:.2f}** | {gru_time:.1f} sn |
| **XGBoost** | {xgb_mse:.0f} | {xgb_rmse:.2f} | {xgb_mae:.2f} | %{xgb_mape:.2f} | **%{xgb_r2*100:.2f}** | {xgb_time:.1f} sn |

**Grafik Yorumu:**
Grafikte net bir şekilde görüldüğü üzere modeller; tatiller, kapalı günler ve kampanyalar sebebiyle oluşan sert zikzakları (trendleri) eklenen dış veriler (Rolling Means, StateHoliday vb.) sayesinde yüksek bir isabetle kavramıştır. Özellikle XGBoost'un tablo verilerindeki (tabular data) hızı ve Derin Öğrenme modellerinin (LSTM/GRU) genel trendleri yakalama yetenekleri projede başarılı bir şekilde sergilenmiştir.
"""

display(Markdown(dynamic_report))


In [ ]:
print("--- OVERFITTING KONTROLÜ ---")
# GRU ve LSTM için son epoch loss değerleri
last_gru_train_loss = gru_tl[-1]
last_gru_val_loss = gru_vl[-1]

last_lstm_train_loss = lstm_tl[-1]
last_lstm_val_loss = lstm_vl[-1]

def check_overfitting(train_loss, val_loss, model_name):
    print(f"\n{model_name} Modeli:")
    print(f"  Son Train Loss: {train_loss:.6f}")
    print(f"  Son Val Loss  : {val_loss:.6f}")
    
    # Val loss train loss'tan çok yüksekse overfit vardır.
    # Bizde Dropout olduğu için Val Loss genellikle Train Loss'tan küçük veya eşit çıkar.
    ratio = val_loss / train_loss
    print(f"  Val/Train Oranı: {ratio:.2f}")
    
    if ratio > 2.0:
        print("  ! UYARI: Validation kaybı Train kaybından çok yüksek. Model ezber (overfitting) yapmış olabilir!")
    elif ratio < 1.0:
        print("  ✓ BAŞARILI: Validation kaybı Train kaybından daha düşük (veya yakın).")
        print("    Bu, modele eklenen Dropout (%30) katmanının çok başarılı bir şekilde ezberi (overfitting) engellediğini gösterir.")
    else:
        print("  ✓ BAŞARILI: Train ve Validation kayıpları birbirine çok yakın. Sağlıklı bir öğrenme gerçekleşmiş, ezber yok.")

check_overfitting(last_gru_train_loss, last_gru_val_loss, "GRU")
check_overfitting(last_lstm_train_loss, last_lstm_val_loss, "LSTM")